In [1]:
import torch
import tiktoken
import gc
from transformers import GPT2Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sd = GPT2Model.from_pretrained("gpt2").to(device).state_dict()
enc = tiktoken.get_encoding("gpt2")

n_layers, n_heads, d_model = 12, 12, 768
d_head = 64

def layer_norm(x, weight, bias, eps=1e-5):
    mean = x.mean(-1, keepdim=True)
    var = ((x - mean) ** 2).mean(-1, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps) * weight + bias

def gelu(x):  # GPT-2 tanh approximation
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x ** 3)))

def raw_forward(tokens, hook_layer=None, start_layer=0, x=None):
    stop = hook_layer if hook_layer else n_layers
    B, seq_len = tokens.shape

    if x is None:
        x = sd['wte.weight'][tokens] + sd['wpe.weight'][:seq_len]

    for layer in range(start_layer, stop):
        ln1_out = layer_norm(x, sd[f'h.{layer}.ln_1.weight'], sd[f'h.{layer}.ln_1.bias'])

        W_qkv = sd[f'h.{layer}.attn.c_attn.weight']
        b_qkv = sd[f'h.{layer}.attn.c_attn.bias']

        q = ln1_out @ W_qkv[:, :768]     + b_qkv[:768]
        k = ln1_out @ W_qkv[:, 768:1536] + b_qkv[768:1536]
        v = ln1_out @ W_qkv[:, 1536:]    + b_qkv[1536:]

        q = q.view(B, seq_len, n_heads, d_head).transpose(1, 2)
        k = k.view(B, seq_len, n_heads, d_head).transpose(1, 2)
        v = v.view(B, seq_len, n_heads, d_head).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / (d_head ** 0.5)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        scores.masked_fill_(mask, float('-inf'))
        pattern = torch.softmax(scores, dim=-1)

        z = (pattern @ v).transpose(1, 2).contiguous().view(B, seq_len, 768)

        x = x + z @ sd[f'h.{layer}.attn.c_proj.weight'] + sd[f'h.{layer}.attn.c_proj.bias']

        ln2_out = layer_norm(x, sd[f'h.{layer}.ln_2.weight'], sd[f'h.{layer}.ln_2.bias'])
        hidden = gelu(ln2_out @ sd[f'h.{layer}.mlp.c_fc.weight'] + sd[f'h.{layer}.mlp.c_fc.bias'])
        x = x + hidden @ sd[f'h.{layer}.mlp.c_proj.weight'] + sd[f'h.{layer}.mlp.c_proj.bias']

    if hook_layer:
        return x

    x = layer_norm(x, sd['ln_f.weight'], sd['ln_f.bias'])
    logits = x @ sd['wte.weight'].T
    return logits

gc.collect()
print(f"GPT-2 small loaded on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


GPT-2 small loaded on cuda


In [3]:
prompt = "The Golden Gate Bridge spans"
token_ids = [50256] + enc.encode(prompt)
generated = list(token_ids)

with torch.no_grad():
    for _ in range(20):
        logits = raw_forward(torch.tensor([generated], device=device))
        generated.append(logits[0, -1].argmax().item())

print(enc.decode(generated))

<|endoftext|>The Golden Gate Bridge spans the San Francisco Bay Area and spans the Bay Area's Pacific Coast. The bridge is the world's


In [4]:
# Raw steering: difference-in-means direction
# direction = mean(concept residuals) - mean(baseline residuals)

hook_layer = 8

concept_prompts = [
    "The Golden Gate Bridge spans the bay",
    "The Golden Gate Bridge was painted international orange",
    "Tourists photograph the Golden Gate Bridge every day",
    "The Golden Gate Bridge towers rise above the fog",
    "Walking across the Golden Gate Bridge takes thirty minutes",
    "The Golden Gate Bridge connects San Francisco to Marin",
    "Ships pass under the Golden Gate Bridge entering the bay",
    "The Golden Gate Bridge opened in nineteen thirty seven",
    "You can see the Golden Gate Bridge from Alcatraz Island",
    "The Golden Gate Bridge is one of the most famous bridges",
]
baseline_prompts = [
    "The best recipe for chocolate cake requires butter",
    "Dogs are popular pets around the world today",
    "The stock market opened higher this morning",
    "Rain is expected tomorrow afternoon in the region",
    "The history of ancient Rome spans many centuries",
    "Basketball players train every day for hours",
    "The piano concert was held in the evening",
    "Farmers harvest wheat in the autumn months",
    "The chemistry experiment produced unexpected results",
    "Children learn to read in elementary school",
]

def get_mean_resid(prompts):
    resids = []
    with torch.no_grad():
        for p in prompts:
            tokens = torch.tensor([[50256] + enc.encode(p)], device=device)
            resid = raw_forward(tokens, hook_layer=hook_layer)
            resids.append(resid[0, -1])
    return torch.stack(resids).mean(dim=0)

direction = get_mean_resid(concept_prompts) - get_mean_resid(baseline_prompts)

def steer(prompt, direction, strength=10.0):
    token_ids = [50256] + enc.encode(prompt)
    generated = list(token_ids)
    with torch.no_grad():
        for _ in range(30):
            tokens = torch.tensor([generated], device=device)
            resid = raw_forward(tokens, hook_layer=hook_layer)
            resid[:, -1] += direction * strength
            logits = raw_forward(tokens, start_layer=hook_layer, x=resid)
            generated.append(logits[0, -1].argmax().item())
    return enc.decode(generated)

lines = []
lines.append(f"No steering:\n{steer('Here is the title of a love story: ', direction, 0.0)}\n")
for s in [1, 2, 3, 4, 5]:
    lines.append(f"Strength={s}:\n{steer('Here is the title of a love story: ', direction, s)}\n")
print("\n".join(lines))

No steering:
<|endoftext|>Here is the title of a love story:  I'm a little bit of a fan of the original series, but I'm not sure if I'm going to be able to enjoy it.

Strength=1:
<|endoftext|>Here is the title of a love story:  "The Great American Trail" by the late John F. Kennedy.  The trail is a beautiful, beautiful, beautiful, beautiful, beautiful

Strength=2:
<|endoftext|>Here is the title of a love story:  "The Story of the City of the Angels" by the Angels of the Bayou.  The Angels of the Bayou is a city

Strength=3:
<|endoftext|>Here is the title of a love story: ia.

The Story of the City of the and in the City of the

City of the

City of the

City

Strength=4:
<|endoftext|>Here is the title of a love story: ia-h-way.

The City of San-head in the County of San-head in the County of San-head in the

Strength=5:
<|endoftext|>Here is the title of a love story:  in the City of San- in- and in--------------------

